<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_valentin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataloader

**Autor:** Valentin · **Datum:** 10.10.2026

**Goal:** A Dataloader that implements
* Byte Pair Tokenization based on TikToken
* Variable Sliding Window
* Token and Position Embeddings using Pytorch nn.embeding

# Result

In [ ]:
text = "Lorem ipsum dolor sit amet, consetetur sadipscing elitr, sed diam nonumy eirmod tempor invidunt ut labore et dolore magna aliquyam erat, sed diam voluptua. At vero eos et accusam et justo duo dolores et ea rebum. Stet clita kasd gubergren, no sea takimata sanctus est Lorem ipsum dolor sit amet. Lorem ipsum dolor sit amet, consetetur sadipscing elitr, sed diam nonumy eirmod tempor invidunt ut labore et dolore magna aliquyam erat, sed diam voluptua. At vero eos et accusam et justo duo dolores et ea rebum. Stet clita kasd gubergren, no sea takimata sanctus est Lorem ipsum dolor sit amet."

contextWindow = 4
output_dim = 128
vocab_size = 50257

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Dataset and Datalaoder

class ContextWindowDataset(Dataset):
  def __init__(self, text:str, tokenizer, contextWindow:int, stride:int) -> None:
    self.inputIds = []
    self.targetIds = []

    # tokenize
    tokenIds = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    assert len(tokenIds) > contextWindow, "Text is too short"

    for i in range(0, len(tokenIds)-contextWindow, stride):
      input = tokenIds[i:i+contextWindow]
      target = tokenIds[i+1:i+contextWindow+1]
      self.inputIds.append(torch.tensor(input))
      self.targetIds.append(torch.tensor(target))

  def __len__(self):
    return len(self.inputIds)

  def __getitem__(self, idx):
    return self.inputIds[idx], self.targetIds[idx]

In [ ]:
def createDataloader(text, contextWindow=4, stride=4, batch_size=8, drop_last=True, shuffle=False):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = ContextWindowDataset(text, tokenizer, contextWindow, stride)
  return DataLoader(dataset, batch_size=batch_size, drop_last=drop_last, shuffle=shuffle)

In [ ]:
dataloader = createDataloader(text)

input, target = next(iter(dataloader))

In [ ]:
positoinEmbeddingLayer = torch.nn.Embedding(contextWindow, output_dim)
tokenEmbedingLayer = torch.nn.Embedding(vocab_size, output_dim)

embeddedToken = tokenEmbedingLayer(input)
embeddedPosition = positoinEmbeddingLayer(torch.arange(contextWindow))

embeddedImput = embeddedToken + embeddedPosition

print(embeddedImput.shape)# batch_size=8, contextWindow=4, output_dim = 128

torch.Size([8, 4, 128])


# Tesing and Understanding

In [ ]:
text = "Once upon a time there was a lovely princess. But she had an enchantment upon her of a fearful sort, which could only be broken by Love's first kiss. She was locked away in a castle guarded by a terrible fire breathing dragon. Many brave knights had attempted to free her from this dreadful prison, but none prevailed. She waited in the dragon's keep in the highest room of the tallest tower for her true love and true love's first kiss. Like that's ever going to happen. What a loony. Shrek Beware Stay out I think he's in here."

In [ ]:
import tiktoken
import re
contextWindow = 4

tokenizer = tiktoken.get_encoding("gpt2")

# Convert Text to tokens(integers)
tokenId = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(tokenId)

In [ ]:
# Seeing the decoded tokens
integersDecoded = "";
for i in tokenId:
  token = tokenizer.decode([i])
  if token[0] == " ":
    integersDecoded += token
  elif re.match(r"[.,!?;:]", token):
    integersDecoded += token
  else:
    integersDecoded += "-" + token

print(integersDecoded)

In [ ]:
import torch
torch.manual_seed(123)
# Sliding window
if len(tokenId) < contextWindow:
  contextWindow = len(tokenId)-1

stride = contextWindow
input_id = []
target_id = []

for i in range(0, len(tokenId)-contextWindow, stride):
  input = tokenId[i:i+contextWindow]
  target = tokenId[i+1:i+contextWindow+1]
  input_id.append(torch.tensor(input))
  target_id.append(torch.tensor(target))

print(input_id)
print(target_id)

In [ ]:
# Dataset and Datalaoder
from torch.utils.data import Dataset, DataLoader

class ContextWindowDataset(Dataset):
  def __init__(self, text:str, tokenizer, contextWindow:int, stride:int) -> None:
    self.inputIds = []
    self.targetIds = []

    # tokenize
    tokenIds = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    assert len(tokenIds) > contextWindow, "Text is too short"

    for i in range(0, len(tokenId)-contextWindow, stride):
      input = tokenIds[i:i+contextWindow]
      target = tokenIds[i+1:i+contextWindow+1]
      self.inputIds.append(torch.tensor(input))
      self.targetIds.append(torch.tensor(target))

  def __len__(self):
    return len(self.inputIds)

  def __getitem__(self, idx):
    return self.inputIds[idx], self.targetIds[idx]

In [ ]:
def createDataloader(text, contextWindow=4, stride=1, batch_size=8, drop_last=True, shuffle=False):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = ContextWindowDataset(text, tokenizer, contextWindow, stride)
  return DataLoader(dataset, batch_size=batch_size, drop_last=drop_last, shuffle=shuffle)

In [ ]:
dataloader = createDataloader(text)

input, target = next(iter(dataloader))
print(input)
print(target)

print(f'input shape: {input.shape}')
print(f'target shape: {target.shape}') #batches = 8, ContextWindow = 4

In [ ]:
# Embedding for Tokens
vocab_size = tokenizer.n_vocab #50257
output_dim = 16

tokenEmbedingLayer = torch.nn.Embedding(vocab_size, output_dim)
print(tokenEmbedingLayer) #vocabular=50257, Embedding-Dimensions=16

In [ ]:
# Embedding for Position
positoinEmbeddingLayer = torch.nn.Embedding(contextWindow, output_dim)
print(positoinEmbeddingLayer) # ContextWindow= 4, Output-Dimentions=16

In [ ]:
embeddedToken = tokenEmbedingLayer(input)
embeddedPosition = positoinEmbeddingLayer(torch.arange(contextWindow))

print(f'embedded_token shape: {embeddedToken.shape}')
print(f'embedded_position shape: {embeddedPosition.shape}')

In [ ]:
embeddedImput = embeddedToken + embeddedPosition
print(embeddedImput.shape)